# 02 — SQL Exploratory Data Analysis

**Objective:** Query `data/insurai.db` using structured SQL (`GROUP BY`, `CASE WHEN`, subqueries) to quantify churn rates, identify demographic and behavioral drivers, and measure annual recurring premium at risk (RM).

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

DB_PATH = Path("..") / "data" / "insurai.db"
conn = sqlite3.connect(DB_PATH)

def run_query(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

## Query 1: Executive Portfolio KPIs
**Objective:** Establish baseline portfolio size, overall churn rate, and total annual recurring premium at risk.

In [2]:
q1 = """
SELECT
    COUNT(*) AS total_policies,
    SUM(churned) AS churned_policies,
    ROUND(AVG(churned) * 100, 2) AS overall_churn_rate_pct,
    ROUND(SUM(annual_value_rm), 2) AS total_annual_premium_rm,
    ROUND(SUM(CASE WHEN churned = 1 THEN annual_value_rm ELSE 0 END), 2) AS annual_revenue_at_risk_rm,
    ROUND(AVG(tenure_months), 1) AS avg_tenure_months
FROM policyholders;
"""
display(run_query(q1))

,total_policies,churned_policies,overall_churn_rate_pct,total_annual_premium_rm,annual_revenue_at_risk_rm,avg_tenure_months
0,1500,431,28.73,3617480.88,1140925.8,31.3


## Query 2: Churn Rate & Revenue at Risk by Region and Policy Type
**Objective:** Pinpoint which regional and product-tier combinations have the highest churn rates and financial exposure.

In [3]:
q2 = """
SELECT
    region,
    policy_type,
    COUNT(*) AS total_customers,
    SUM(churned) AS churned_customers,
    ROUND(AVG(churned) * 100, 2) AS churn_rate_pct,
    ROUND(SUM(CASE WHEN churned = 1 THEN annual_value_rm ELSE 0 END), 2) AS revenue_at_risk_rm
FROM policyholders
GROUP BY region, policy_type
ORDER BY churn_rate_pct DESC;
"""
display(run_query(q2))

,region,policy_type,total_customers,churned_customers,churn_rate_pct,revenue_at_risk_rm
0,Kuala Lumpur,Investment-Linked,59,30,50.85,142931.40
1,Johor,Investment-Linked,50,25,50.00,113698.20
2,Selangor,Investment-Linked,70,27,38.57,125582.28
3,Kuala Lumpur,Medical & Health,150,57,38.00,154871.04
4,Johor,Life Protection,69,24,34.78,46498.20
5,Kuala Lumpur,Life Protection,96,29,30.21,58830.96
6,Johor,Medical & Health,63,17,26.98,43009.32
7,Selangor,Motor Comprehensive,139,37,26.62,55629.12
8,Selangor,Medical & Health,130,34,26.15,88399.56
9,Penang,Investment-Linked,16,4,25.00,15969.48


## Query 3: Tenure Cohort Churn Analysis
**Objective:** Test whether newer policyholders (`< 12` months tenure) churn at higher rates than established customers.

In [4]:
q3 = """
SELECT
    CASE
        WHEN tenure_months < 12 THEN '01. New (< 12 mo)'
        WHEN tenure_months BETWEEN 12 AND 24 THEN '02. Early (12-24 mo)'
        WHEN tenure_months BETWEEN 25 AND 48 THEN '03. Mid (25-48 mo)'
        ELSE '04. Loyal (49+ mo)'
    END AS tenure_cohort,
    COUNT(*) AS total_customers,
    SUM(churned) AS churned_customers,
    ROUND(AVG(churned) * 100, 2) AS churn_rate_pct,
    ROUND(SUM(CASE WHEN churned = 1 THEN annual_value_rm ELSE 0 END), 2) AS revenue_at_risk_rm
FROM policyholders
GROUP BY tenure_cohort
ORDER BY tenure_cohort;
"""
display(run_query(q3))

,tenure_cohort,total_customers,churned_customers,churn_rate_pct,revenue_at_risk_rm
0,01. New (< 12 mo),212,130,61.32,322128.96
1,02. Early (12-24 mo),458,120,26.20,351107.88
2,03. Mid (25-48 mo),580,127,21.90,326471.64
3,04. Loyal (49+ mo),250,54,21.60,141217.32


## Query 4: Late Payments & Support Tickets Impact
**Objective:** Quantify how billing friction (`late_payments`) and service friction (`support_tickets`) drive churn.

In [5]:
q4_late = """
SELECT
    CASE
        WHEN late_payments = 0 THEN '0 Late Payments'
        WHEN late_payments = 1 THEN '1 Late Payment'
        ELSE '2+ Late Payments'
    END AS payment_status,
    COUNT(*) AS total_customers,
    SUM(churned) AS churned_customers,
    ROUND(AVG(churned) * 100, 2) AS churn_rate_pct
FROM policyholders
GROUP BY payment_status
ORDER BY churn_rate_pct ASC;
"""
display(run_query(q4_late))

q4_tickets = """
SELECT
    CASE
        WHEN support_tickets = 0 THEN '0 Tickets'
        WHEN support_tickets = 1 THEN '1 Ticket'
        ELSE '2+ Tickets'
    END AS ticket_bucket,
    COUNT(*) AS total_customers,
    SUM(churned) AS churned_customers,
    ROUND(AVG(churned) * 100, 2) AS churn_rate_pct
FROM policyholders
GROUP BY ticket_bucket
ORDER BY churn_rate_pct ASC;
"""
display(run_query(q4_tickets))

,payment_status,total_customers,churned_customers,churn_rate_pct
0,0 Late Payments,757,158,20.87
1,1 Late Payment,536,161,30.04
2,2+ Late Payments,207,112,54.11


,ticket_bucket,total_customers,churned_customers,churn_rate_pct
0,0 Tickets,721,156,21.64
1,1 Ticket,520,158,30.38
2,2+ Tickets,259,117,45.17


## Query 5: Churn Reason Breakdown & High-Value Churned Policies (Subquery)
**Objective:** Identify the primary stated reasons for cancellation and isolate churned policies whose annual value exceeds the portfolio average.

In [6]:
q5_reasons = """
SELECT
    churn_reason,
    COUNT(*) AS churned_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM policyholders WHERE churned = 1), 2) AS share_of_churn_pct,
    ROUND(SUM(annual_value_rm), 2) AS lost_annual_value_rm
FROM policyholders
WHERE churned = 1
GROUP BY churn_reason
ORDER BY churned_count DESC;
"""
display(run_query(q5_reasons))

q5_high_value = """
SELECT
    policy_id,
    region,
    policy_type,
    tenure_months,
    annual_value_rm,
    late_payments,
    support_tickets,
    churn_reason
FROM policyholders
WHERE churned = 1
  AND annual_value_rm > (SELECT AVG(annual_value_rm) FROM policyholders)
ORDER BY annual_value_rm DESC
LIMIT 10;
"""
display(run_query(q5_high_value))

,churn_reason,churned_count,share_of_churn_pct,lost_annual_value_rm
0,Premium too expensive,142,32.95,401702.52
1,Poor customer service,92,21.35,231867.12
2,Competitor offered better price,59,13.69,159715.32
3,Claim rejected or dissatisfied,54,12.53,151868.40
4,No longer needs coverage,53,12.30,121726.32
5,Relocated or moved abroad,31,7.19,74046.12


,policy_id,region,policy_type,tenure_months,annual_value_rm,late_payments,support_tickets,churn_reason
0,POL-10881,Kuala Lumpur,Investment-Linked,8,7547.16,0,0,Poor customer service
1,POL-11497,Kuala Lumpur,Investment-Linked,61,6826.20,2,1,Premium too expensive
2,POL-10108,Sarawak,Investment-Linked,26,6789.96,1,1,Premium too expensive
3,POL-10854,Johor,Investment-Linked,11,6744.84,0,2,Premium too expensive
4,POL-10950,Sarawak,Investment-Linked,64,6744.72,0,0,Relocated or moved abroad
5,POL-11423,Johor,Investment-Linked,18,6681.36,0,1,Poor customer service
6,POL-10621,Kuala Lumpur,Investment-Linked,33,6670.56,0,0,Premium too expensive
7,POL-10702,Selangor,Investment-Linked,19,6651.12,3,1,Premium too expensive
8,POL-10622,Selangor,Investment-Linked,16,6438.48,0,1,Premium too expensive
9,POL-10690,Selangor,Investment-Linked,21,6343.56,1,4,Premium too expensive


In [7]:
conn.close()

## Key Takeaways

- **Early-tenure risk:** Policyholders in their first 12 months exhibit substantially higher churn than multi-year customers, making onboarding and first-renewal outreach critical.
- **Behavioral signals:** Both `2+` late payments and `2+` support tickets sharply increase churn probability, giving retention teams clear early-warning triggers.
- **Product & Pricing:** `Investment-Linked` policies carry the highest monthly/annual premiums, so even moderate churn in that tier drives a disproportionate share of `annual_value_rm` at risk.